# SZ3 + model  vs  SPERR — PSNR vs Compression Ratio

Compare our **SZ3 + neural-residual** pipeline against **SPERR** (a wavelet compressor) across four datasets, one figure per dataset:

| Dataset | Target field(s) | Model params | Aux input |
|---|---|---|---|
| NYX 512³ | baryon density, temperature, dark matter density | 30k | other NYX fields |
| Miranda 1024³ | single field | 240k | — |
| Hurricane 100×500×500 | CLOUD | 6k | PRECIP,P,TC,U,V,W |
| Magnetic 512³ | single field | 30k | — |

Each curve sweeps the error bound to span a range of CR.  **SZ3 + model** stores the target's SZ3 stream + the model (bf16); **aux fields are raw side-info, not charged to CR** (NYX/Hurricane convention).  **SPERR** is standalone (its own bitstream) matched to each SZ3 point's PSNR (compare CR at equal quality); its PSNR is recomputed from the decompressed volume with the same formula.

In [1]:
import os, sys, time, subprocess
import numpy as np
import matplotlib.pyplot as plt
import torch

SPERR_BIN    = "/home/sam/Halo_Finder/SPERR/build/bin/sperr3d"
SZ3_LIB      = "/home/sam/Data_Compression/SZ3/build/lib64/libSZ3c.so"
PYSZ_PATH    = "/home/sam/Data_Compression/SZ3/tools/pysz"
SCRIPTS_PATH = "/home/sam/Halo_Finder/Final_design/base_script"
for p in (PYSZ_PATH, SCRIPTS_PATH):
    if p not in sys.path:
        sys.path.append(p)

import importlib, bg_stage
importlib.reload(bg_stage)
from pysz import SZ
from bg_stage import train_bg_only, run_bg_inference, unwrap_bg_model
from experiment import build_bg_only_cfg, estimate_bg_model_param_bytes
from bg_shard import pick_bg_h_under_budget

device    = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
sz_engine = SZ(SZ3_LIB)
BYTES_PER_PARAM = 2          # model weights stored as bf16

def compute_psnr(x_true, x_hat, drange):
    mse = float(np.mean((np.asarray(x_true, np.float64) - np.asarray(x_hat, np.float64)) ** 2))
    return 100.0 if mse == 0 else 20.0 * np.log10(drange) - 10.0 * np.log10(mse)

def bg_h_for_params(budget, shape, n_fields):
    """Largest bg_h whose split-bands model fits the param budget."""
    h, est = pick_bg_h_under_budget(int(budget), shape=shape, n_fields=int(n_fields),
                                    bg_arch="spatial", h_candidates=list(range(3, 256)))
    return int(h)

def run_sperr(data_file, target_gt, shape, drange, target_psnr):
    """SPERR at a target PSNR.  Returns (CR, PSNR, recon, nbytes): the decompressed
    volume (for the SPERR+model curve) and the bitstream size.  PSNR is recomputed."""
    W, H, D = shape[2], shape[1], shape[0]          # SPERR wants fastest-varying first
    tag = np.random.randint(1 << 30)
    bit = f"/tmp/sperr_{tag}.bit"; rec = f"/tmp/sperr_{tag}.dec.f32"
    subprocess.run([SPERR_BIN, "-c", "--ftype", "32",
                    "--dims", str(W), str(H), str(D), "--psnr", f"{float(target_psnr):.4f}",
                    "--bitstream", bit, data_file], capture_output=True, text=True)
    if not os.path.exists(bit):
        return None, None, None, None
    nbytes = os.path.getsize(bit)
    subprocess.run([SPERR_BIN, "-d", "--decomp_f", rec, bit], capture_output=True, text=True)
    cr = psnr = recon = None
    if os.path.exists(rec):
        recon = np.fromfile(rec, dtype=np.float32).reshape(shape)
        psnr  = compute_psnr(target_gt, recon, drange)
        cr    = (int(np.prod(shape)) * 4) / nbytes
    for f in (bit, rec):
        if os.path.exists(f):
            os.remove(f)
    return cr, psnr, recon, nbytes

print("Setup ready | device:", device)

Setup ready | device: cuda:0


In [2]:
def _fmt(x):
    return f"{x:6.1f}" if x is not None else "  n/a "

def bench_field(name, target_gt, target_file, aux_list, shape, rel_errs,
                param_budget, epochs, lr=1e-3, sperr_psnr_offset=10.0,
                sperr_extra_span=0.0, sperr_n_extra=0):
    """Run SZ3, SPERR, and SZ3+model across rel_errs for one target field.
    aux_list = raw (uncompressed) side-info fields; NOT charged to CR."""
    target_gt = np.asarray(target_gt, np.float32)
    drange    = float(target_gt.max() - target_gt.min())
    n_fields  = 1 + len(aux_list)
    bg_h      = bg_h_for_params(param_budget, shape, n_fields)
    n_params, nn_bytes = estimate_bg_model_param_bytes(
        n_fields=n_fields, shape=shape, bg_arch="spatial", bg_h=bg_h,
        dtype_bytes=BYTES_PER_PARAM)
    depth, patch = shape[0], shape[2]
    orig_bytes   = int(np.prod(shape)) * 4
    print(f"[{name}] budget {param_budget:,} -> bg_h={bg_h} (~{n_params:,} params, "
          f"{nn_bytes/1e3:.1f} KB) | n_fields={n_fields} | drange={drange:.3g}")
    neurlz_features = None
    if ADD_NEURLZ:
        neurlz_features = (_neurlz_features_for_params(n_params, n_fields)
                           if NEURLZ_FEATURES == "match" else NEURLZ_FEATURES)
        print(f"[{name}] NeurLZ BasicUNet features={tuple(neurlz_features)} "
              f"(~{_basicunet_nparams(neurlz_features, n_fields):,} params)")

    sz3   = {"CR": [], "PSNR": []}; pipe       = {"CR": [], "PSNR": []}
    sperr = {"CR": [], "PSNR": []}; sperr_pipe = {"CR": [], "PSNR": []}
    neurlz = {"CR": [], "PSNR": []}

    # Train the neural residual model on a given base reconstruction; the output is
    # clamped within `base_rel` * range of the base (like the SZ3 pipeline does).
    def _train_residual(base_recon, base_rel):
        Xs  = [target_gt] + aux_list
        Xps = [np.ascontiguousarray(base_recon, np.float32)] + aux_list
        cfg = build_bg_only_cfg(
            X_target=Xs[0], Xps=Xps, max_train_time=1e9, bg_h=bg_h, roi_h=4,
            epochs=epochs, steps_per_epoch=depth, bg_patch_size=patch, bg_batch=1, lr=lr,
            bg_freq_weight=1.0, bg_fft_phase_weight=1.0, bg_freq_warmup_epochs=1,
            bg_field_norm="zscore")
        cfg.bg_arch = "spatial"; cfg.bg_split_mode = "three"; cfg.bg_split_bands = True
        cfg.bg_split_sigma = 0.12; cfg.bg_sigma_low = 0.08; cfg.bg_sigma_mid = 0.18
        cfg.bg_cr_rel_err = float(base_rel)
        cfg.bg_gpu_sampling = True
        def ev(model, c=cfg, Xs=Xs, Xps=Xps, r=base_rel):
            return compute_psnr(Xs[0], run_bg_inference(model, Xs, Xps, c, float(r)), drange), 0.0
        model, _ = train_bg_only(Xs=Xs, Xps=Xps, device=device, cfg=cfg, evaluator=ev)
        p = compute_psnr(Xs[0], run_bg_inference(model, Xs, Xps, cfg, float(base_rel)), drange)
        del model
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
        return p

    # ── SZ3  and  SZ3 + model ── (rel sweep; CR moves with the SZ3 bound)
    for rel in rel_errs:
        b, _   = sz_engine.compress(target_gt, 1, 0, float(rel), 0)
        sz_len = len(b)
        xq     = sz_engine.decompress(b, shape, np.float32)
        p_sz3  = compute_psnr(target_gt, xq, drange); cr_sz3 = orig_bytes / sz_len
        sz3["CR"].append(cr_sz3); sz3["PSNR"].append(p_sz3)
        p_pipe  = _train_residual(xq, rel)
        cr_pipe = orig_bytes / (sz_len + nn_bytes)
        pipe["CR"].append(cr_pipe); pipe["PSNR"].append(p_pipe)
        print(f"  rel={rel:.0e} | SZ3 {cr_sz3:6.1f}x/{p_sz3:5.1f}dB | "
              f"SZ3+model {cr_pipe:6.1f}x/{p_pipe:5.1f}dB (+{p_pipe-p_sz3:.1f})")
        if ADD_NEURLZ:
            p_nlz, nlz_params = run_neurlz(target_gt, xq, aux_list, shape, drange, rel,
                                           int(epochs * NEURLZ_EPOCH_MULT), neurlz_features)
            cr_nlz = orig_bytes / (sz_len + nlz_params * BYTES_PER_PARAM)
            neurlz["CR"].append(cr_nlz); neurlz["PSNR"].append(p_nlz)
            print(f"           SZ3+NeurLZ {cr_nlz:6.1f}x/{p_nlz:5.1f}dB (+{p_nlz-p_sz3:.1f}) [{nlz_params:,}p]")

    # ── SPERR  and  SPERR + model ── (target-PSNR sweep; lower PSNR -> higher CR)
    if sz3["PSNR"]:
        hi = max(sz3["PSNR"]) - sperr_psnr_offset
        lo = min(sz3["PSNR"]) - sperr_psnr_offset - sperr_extra_span
        n_pts = len(rel_errs) + sperr_n_extra
        print(f"  SPERR sweep: {n_pts} targets, PSNR {hi:.1f}..{lo:.1f} dB")
        for tp in np.linspace(hi, lo, n_pts):
            cr_sp, p_sp, recon_sp, sp_bytes = run_sperr(target_file, target_gt, shape, drange, float(tp))
            if recon_sp is None:
                continue
            sperr["CR"].append(cr_sp); sperr["PSNR"].append(p_sp)
            # clamp band = SPERR's own L-inf relative error (analogue of SZ3's rel)
            rel_sp  = float(np.abs(target_gt - recon_sp).max()) / max(drange, 1e-12)
            p_spp   = _train_residual(recon_sp, rel_sp)
            cr_spp  = orig_bytes / (sp_bytes + nn_bytes)
            sperr_pipe["CR"].append(cr_spp); sperr_pipe["PSNR"].append(p_spp)
            print(f"    SPERR {cr_sp:6.1f}x/{p_sp:5.1f}dB | SPERR+model {cr_spp:6.1f}x/{p_spp:5.1f}dB "
                  f"(+{p_spp-p_sp:.1f})")
            del recon_sp
            torch.cuda.empty_cache() if torch.cuda.is_available() else None
        for D in (sperr, sperr_pipe):
            o = list(np.argsort(D["CR"]))
            D["CR"] = [D["CR"][i] for i in o]; D["PSNR"] = [D["PSNR"][i] for i in o]

    return dict(sz3=sz3, pipe=pipe, sperr=sperr, sperr_pipe=sperr_pipe, neurlz=neurlz,
                bg_h=bg_h, n_params=n_params)

def plot_panel(ax, r, title):
    ax.plot(r["sz3"]["CR"], r["sz3"]["PSNR"], "^--", color="tab:gray", label="SZ3")
    ax.plot(r["pipe"]["CR"], r["pipe"]["PSNR"], "o-", color="tab:orange",
            label=f"SZ3 + model ({r['n_params']/1000:.0f}k)")
    ax.plot(r["sperr"]["CR"], r["sperr"]["PSNR"], "s--", color="tab:blue", label="SPERR")
    ax.plot(r["sperr_pipe"]["CR"], r["sperr_pipe"]["PSNR"], "D-", color="tab:green",
            label="SPERR + model")
    if r.get("neurlz", {}).get("CR"):
        ax.plot(r["neurlz"]["CR"], r["neurlz"]["PSNR"], "v:", color="tab:red", label="SZ3 + NeurLZ")
    ax.set_xlabel("Compression Ratio"); ax.set_ylabel("PSNR (dB)")
    ax.set_title(title); ax.grid(True, alpha=0.3); ax.legend(fontsize=9)

print("bench_field ready")

bench_field ready


##### NeurLZ


In [3]:
# ── NeurLZ baseline (SZ3 + monai BasicUNet error predictor) ──────────────────
# Faithful re-impl of neurlz/train.py's recipe (per-slice BasicUNet predicts the
# minmax-normalized SZ3 error; L1, Adam 1e-2, cosine T_max=1500); enhanced output
# clamped to the rel error bound (same as the SZ3+model pipeline).
import io, contextlib, random
from monai.networks.nets import BasicUNet
from config_io import _error_bounded_post_process

ADD_NEURLZ        = True
NEURLZ_LR         = 1e-2
NEURLZ_BATCH      = 10
NEURLZ_MAX_PIXELS_PER_BATCH = 1024 * 1024   # cap batch*H*W so big slices don't OOM
NEURLZ_VERBOSE    = True     # print per-epoch loss + PSNR
NEURLZ_EVAL_EVERY = 1        # compute PSNR every K epochs (loss prints every epoch)
NEURLZ_EPOCH_MULT = 1
# BasicUNet channel widths (6-tuple). "match" auto-sizes a uniform (w,)*6 so its
# param count ~= your BG model's -> iso-parameter / iso-CR comparison on the plot.
# Or give an explicit tuple, e.g. (4,4,4,4,4,4) for NeurLZ's default (~3k params).
NEURLZ_FEATURES   = "match"

def _basicunet_nparams(features, n_fields):
    with contextlib.redirect_stdout(io.StringIO()):
        m = BasicUNet(spatial_dims=2, features=tuple(features), act="gelu",
                      in_channels=int(n_fields), out_channels=1)
    n = sum(p.numel() for p in m.parameters() if p.requires_grad); del m
    return n

def _neurlz_features_for_params(target_params, n_fields, lo=4, hi=384):
    """Uniform width (w,)*6 BasicUNet whose param count is closest to target_params."""
    a, b, best = lo, hi, lo
    while a <= b:
        mid = (a + b) // 2
        if _basicunet_nparams((mid,) * 6, n_fields) <= target_params:
            best = mid; a = mid + 1
        else:
            b = mid - 1
    w_hi = min(best + 1, hi)
    n_lo = _basicunet_nparams((best,) * 6, n_fields)
    n_hi = _basicunet_nparams((w_hi,) * 6, n_fields)
    w = best if abs(n_lo - target_params) <= abs(n_hi - target_params) else w_hi
    return (w,) * 6

def _mm(x, eps=1e-8):
    lo, hi = float(np.min(x)), float(np.max(x))
    return ((np.asarray(x, np.float32) - lo) / (hi - lo + eps)).astype(np.float32), (lo, hi)

def run_neurlz(target_gt, base_recon, aux_list, shape, drange, rel, epochs, features):
    """SZ3 + NeurLZ on one base reconstruction. Returns (psnr, n_params)."""
    D, H, W = shape
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    eff_batch = max(1, min(NEURLZ_BATCH, NEURLZ_MAX_PIXELS_PER_BATCH // (H * W)))
    if eff_batch < NEURLZ_BATCH:
        print(f"           [neurlz] slice {H}x{W}: batch {NEURLZ_BATCH} -> {eff_batch} (memory)")
    tgt = np.asarray(target_gt, np.float32)
    lq  = np.ascontiguousarray(base_recon, np.float32)
    fields = [lq] + [np.asarray(a, np.float32) for a in aux_list]
    n_fields = len(fields)
    lq_n = np.stack([_mm(f)[0] for f in fields], axis=1)
    err_n, (e_lo, e_hi) = _mm(tgt - lq)
    ph, pw = (-H) % 16, (-W) % 16
    pad = ((0, 0), (0, 0), (0, ph), (0, pw))
    Xlq  = torch.from_numpy(np.pad(lq_n, pad, mode="reflect"))
    Yerr = torch.from_numpy(np.pad(err_n[:, None], pad, mode="reflect"))

    torch.manual_seed(17); np.random.seed(17); random.seed(17)
    with contextlib.redirect_stdout(io.StringIO()):
        model = BasicUNet(spatial_dims=2, features=tuple(features), act="gelu",
                          in_channels=n_fields, out_channels=1).to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    opt   = torch.optim.Adam(model.parameters(), lr=NEURLZ_LR)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=1500)
    mse    = torch.nn.MSELoss()
    idx = np.arange(D)
    def _enhanced():
        model.eval()
        out = lq.copy()
        with torch.no_grad():
            for st in range(0, D, eff_batch):
                bi = list(range(st, min(st + eff_batch, D)))
                pred = model(Xlq[bi].to(device)).cpu().numpy()[:, 0, :H, :W]
                out[bi] = lq[bi] + (pred * (e_hi - e_lo + 1e-8) + e_lo)
        model.train()
        return _error_bounded_post_process(x_enhanced=out, x_prime=lq, absolute_error_bound=0.0,
                                           relative_error_bound=float(rel), verbose=False, a=1.0)

    model.train()
    for ep in range(int(epochs)):
        np.random.shuffle(idx)
        tot, nb = 0.0, 0
        for st in range(0, D, eff_batch):
            bi = idx[st:st + eff_batch]
            loss = mse(model(Xlq[bi].to(device)), Yerr[bi].to(device))
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
            tot += float(loss.item()); nb += 1
        if NEURLZ_VERBOSE:
            avg = tot / max(nb, 1)
            if (ep + 1) % NEURLZ_EVAL_EVERY == 0 or ep == int(epochs) - 1:
                pe = compute_psnr(tgt, _enhanced(), drange)
                print(f"           [neurlz] epoch {ep+1:3d}/{int(epochs)} | MSE {avg:.6f} | PSNR {pe:.2f} dB")
            else:
                print(f"           [neurlz] epoch {ep+1:3d}/{int(epochs)} | MSE {avg:.6f}")

    enh = _enhanced()
    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return compute_psnr(tgt, enh, drange), int(n_params)

print("run_neurlz ready | ADD_NEURLZ =", ADD_NEURLZ, "| NEURLZ_FEATURES =", NEURLZ_FEATURES)


/home/sam/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


run_neurlz ready | ADD_NEURLZ = True | NEURLZ_FEATURES = match


## 1. NYX 512³  —  baryon density, temperature, dark matter density (30k params, +aux)

In [ ]:
NYX_DIR   = "/home/sam/Halo_Finder/halo_finder_v1/SDRBENCH-EXASKY-NYX-512x512x512/origin/"
NYX_SHAPE = (512, 512, 512)
NYX_ALL   = ["baryon_density", "dark_matter_density", "temperature",
             "velocity_x", "velocity_y", "velocity_z"]
NYX_TARGETS = ["baryon_density", "temperature", "dark_matter_density"]
# Per-field rel_err: baryon density needs much tighter bounds than temperature /
# dark matter density (different dynamic ranges / compressibility).
NYX_REL = {
    "baryon_density":      [1e-6, 3e-6, 5e-6, 7e-6, 9e-6],
    "temperature":         [1e-4, 3e-4, 5e-4, 7e-4, 9e-4],
    "dark_matter_density": [1e-4, 3e-4, 5e-4, 7e-4, 9e-4],
}
NYX_PARAMS, NYX_EPOCHS, NYX_SPERR_OFF = 30000, 10, 20.0
NYX_SPERR_EXTRA_SPAN, NYX_SPERR_NEXTRA = 15.0, 0   # 5 rel + 2 extra = 7 SPERR points

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, tname in enumerate(NYX_TARGETS):
    tgt = np.fromfile(NYX_DIR + tname + ".f32", dtype=np.float32).reshape(NYX_SHAPE)
    aux = [np.memmap(NYX_DIR + a + ".f32", dtype=np.float32, mode="r", shape=NYX_SHAPE)
           for a in NYX_ALL if a != tname]
    r = bench_field(f"NYX/{tname}", tgt, NYX_DIR + tname + ".f32", aux, NYX_SHAPE,
                    NYX_REL[tname], NYX_PARAMS, NYX_EPOCHS, sperr_psnr_offset=NYX_SPERR_OFF,
                    sperr_extra_span=NYX_SPERR_EXTRA_SPAN, sperr_n_extra=NYX_SPERR_NEXTRA)
    plot_panel(axes[i], r, f"NYX 512 — {tname}")
    del tgt, aux
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

fig.suptitle("NYX 512³ — SZ3+model vs SPERR  (30k params, + aux fields)", fontsize=14)
plt.tight_layout()
plt.savefig("sperr_cmp_nyx.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: sperr_cmp_nyx.png")

## A/B — fp32 training (`cfg.amp=False`) vs bf16 AMP

Same data / params / epochs / seed — **only the AMP dtype differs** — on the tightest-bound (highest-PSNR) NYX point, where bf16's ~7-bit mantissa is most likely to cap precision. Reports the PSNR gain from fp32 and its wall-clock cost. Change `AB_FIELD` / `AB_REL` to probe other operating points (e.g. `baryon_density` @ `1e-6` is even higher-PSNR).

In [ ]:
# ============================================================================
# A/B: does fp32 training (cfg.amp=False) beat bf16 AMP on PSNR?
# Only cfg.amp differs; train_bg_only seeds internally (cfg.seed) so init +
# sampling are identical between the two runs -> the delta is purely fp32 vs bf16.
# Uses final-epoch weights (evaluator=None) for speed; the relative gap is what matters.
# ============================================================================
import time

AB_FIELD  = "baryon_density"   # try "baryon_density" + AB_REL=1e-6 for an even higher-PSNR point
AB_REL    = 5e-6                    # tightest bound for this field -> highest PSNR -> most precision-limited
AB_PARAMS = NYX_PARAMS             # 30k
AB_EPOCHS = NYX_EPOCHS             # 10
AB_USE_AUX = True                  # match the main NYX run (aux fields as extra input channels)

tgt = np.fromfile(NYX_DIR + AB_FIELD + ".f32", dtype=np.float32).reshape(NYX_SHAPE)
aux = ([np.memmap(NYX_DIR + a + ".f32", dtype=np.float32, mode="r", shape=NYX_SHAPE)
        for a in NYX_ALL if a != AB_FIELD] if AB_USE_AUX else [])
drange   = float(tgt.max() - tgt.min())
n_fields = 1 + len(aux)
bg_h     = bg_h_for_params(AB_PARAMS, NYX_SHAPE, n_fields)
n_params, nn_bytes = estimate_bg_model_param_bytes(
    n_fields=n_fields, shape=NYX_SHAPE, bg_arch="spatial", bg_h=bg_h, dtype_bytes=BYTES_PER_PARAM)
depth, patch = NYX_SHAPE[0], NYX_SHAPE[2]
orig_bytes   = int(np.prod(NYX_SHAPE)) * 4

b, _   = sz_engine.compress(tgt, 1, 0, float(AB_REL), 0)
sz_len = len(b)
xq     = sz_engine.decompress(b, NYX_SHAPE, np.float32)
p_sz3  = compute_psnr(tgt, xq, drange)
cr_pipe = orig_bytes / (sz_len + nn_bytes)
print(f"[A/B] {AB_FIELD} rel={AB_REL:.0e} | bg_h={bg_h} (~{n_params:,}p) | "
      f"SZ3 base PSNR={p_sz3:.2f} dB | pipe CR~{cr_pipe:.1f}x | aux={len(aux)} | epochs={AB_EPOCHS}")

def _train_amp(use_amp):
    Xs  = [tgt] + aux
    Xps = [np.ascontiguousarray(xq, np.float32)] + aux
    cfg = build_bg_only_cfg(
        X_target=Xs[0], Xps=Xps, max_train_time=1e9, bg_h=bg_h, roi_h=4,
        epochs=AB_EPOCHS, steps_per_epoch=depth, bg_patch_size=patch, bg_batch=1, lr=1e-3,
        bg_freq_weight=1.0, bg_fft_phase_weight=1.0, bg_freq_warmup_epochs=1,
        bg_field_norm="zscore")
    cfg.bg_arch = "spatial"; cfg.bg_split_mode = "three"; cfg.bg_split_bands = True
    cfg.bg_split_sigma = 0.12; cfg.bg_sigma_low = 0.08; cfg.bg_sigma_mid = 0.18
    cfg.bg_gpu_sampling = True
    cfg.amp = bool(use_amp)                       # <-- the only knob that differs
    t0 = time.time()
    model, _ = train_bg_only(Xs=Xs, Xps=Xps, device=device, cfg=cfg, evaluator=None)
    xh = run_bg_inference(unwrap_bg_model(model), Xs, Xps, cfg, float(AB_REL))
    p  = compute_psnr(Xs[0], xh, drange)
    wall = time.time() - t0
    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return p, wall

p_bf16, t_bf16 = _train_amp(True)
p_fp32, t_fp32 = _train_amp(False)

print("\n========== fp32 vs bf16 (same seed / params / epochs) ==========")
print(f"  SZ3 base           : {p_sz3:7.2f} dB")
print(f"  SZ3+model (bf16)   : {p_bf16:7.2f} dB  (+{p_bf16 - p_sz3:5.2f} over SZ3)  [{t_bf16:.0f}s]")
print(f"  SZ3+model (fp32)   : {p_fp32:7.2f} dB  (+{p_fp32 - p_sz3:5.2f} over SZ3)  [{t_fp32:.0f}s]")
print(f"  >> fp32 - bf16     : {p_fp32 - p_bf16:+.3f} dB   (PSNR gain from fp32 training)")
print(f"  >> fp32 slowdown   : {t_fp32 / max(t_bf16, 1e-9):.2f}x wall")
del tgt, aux, xq
torch.cuda.empty_cache() if torch.cuda.is_available() else None


## 2. Miranda 1024³  —  single field (240k params)

In [ ]:
MIR_FILE  = "/home/sam/Halo_Finder/halo_finder_v1/miranda_1024x1024x1024_float32.raw"
MIR_SHAPE = (1024, 1024, 1024)
MIR_REL   = [1e-3, 3e-3, 5e-3, 7e-3, 9e-3]
MIR_PARAMS, MIR_EPOCHS = 240000, 5

mir = np.fromfile(MIR_FILE, dtype=np.float32).reshape(MIR_SHAPE)   # ~4 GB in RAM
r = bench_field("Miranda", mir, MIR_FILE, [], MIR_SHAPE, MIR_REL, MIR_PARAMS, MIR_EPOCHS)
del mir
torch.cuda.empty_cache() if torch.cuda.is_available() else None

fig, ax = plt.subplots(figsize=(7, 5.5))
plot_panel(ax, r, "Miranda 1024³ — single field")
fig.suptitle("Miranda 1024³ — SZ3+model vs SPERR  (240k params)", fontsize=13)
plt.tight_layout()
plt.savefig("sperr_cmp_miranda.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: sperr_cmp_miranda.png")


[Model: spatial] Total Params: 724
 [Params] Main (BG) Network : 724 parameters

[Model: spatial] Total Params: 1,216
 [Params] Main (BG) Network : 1,216 parameters

[Model: spatial] Total Params: 1,834
 [Params] Main (BG) Network : 1,834 parameters

[Model: spatial] Total Params: 2,578
 [Params] Main (BG) Network : 2,578 parameters

[Model: spatial] Total Params: 3,448
 [Params] Main (BG) Network : 3,448 parameters

[Model: spatial] Total Params: 4,444
 [Params] Main (BG) Network : 4,444 parameters

[Model: spatial] Total Params: 5,566
 [Params] Main (BG) Network : 5,566 parameters

[Model: spatial] Total Params: 6,814
 [Params] Main (BG) Network : 6,814 parameters

[Model: spatial] Total Params: 8,188
 [Params] Main (BG) Network : 8,188 parameters

[Model: spatial] Total Params: 9,688
 [Params] Main (BG) Network : 9,688 parameters

[Model: spatial] Total Params: 11,314
 [Params] Main (BG) Network : 11,314 parameters

[Model: spatial] Total Params: 13,066
 [Params] Main (BG) Network 

/home/sam/Halo_Finder/Final_design/base_script/bg_stage.py:520: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp, dtype=autocast_dtype):


Epoch   1 [BG] | train_wall=24.13s | Loss: 1.555500 | Freq: 2.147987 | Low: 0.053828 | Mid: 0.073431 | High: 0.579796 | Global: 69.27 dB | MaxErr: 0.0  [New Best!]
[timing] first_epoch_pure_train≈24.701s (excludes this epoch's end-of-epoch eval)
Epoch   2 [BG] | train_wall=21.97s | Loss: 2.121780 | Freq: 1.104782 | Low: 0.059834 | Mid: 0.069955 | High: 0.340171 | Global: 70.41 dB | MaxErr: 0.0  [New Best!]
Epoch   3 [BG] | train_wall=21.99s | Loss: 1.941561 | Freq: 1.025291 | Low: 0.052792 | Mid: 0.063859 | High: 0.289510 | Global: 70.45 dB | MaxErr: 0.0  [New Best!]
Epoch   4 [BG] | train_wall=22.00s | Loss: 1.904334 | Freq: 1.009945 | Low: 0.051032 | Mid: 0.061961 | High: 0.279708 | Global: 70.48 dB | MaxErr: 0.0  [New Best!]
Epoch   5 [BG] | train_wall=22.00s | Loss: 1.879070 | Freq: 0.990391 | Low: 0.049557 | Mid: 0.061622 | High: 0.278082 | Global: 70.53 dB | MaxErr: 0.0  [New Best!]

--- Experiment [BG_only] finished ---
--- Pure training time: 112.65 s ---
[timing] epochs=5 | tr

## 3. Hurricane 100×500×500  —  CLOUD (6k params, +aux: PRECIP,P,TC,U,V,W)

In [ ]:
HUR_DIR   = "/home/sam/Halo_Finder/halo_finder_v1/100x500x500/"
HUR_SHAPE = (100, 500, 500)
HUR_TARGET = "CLOUDf48"
HUR_AUX    = ["PRECIPf48", "Pf48", "TCf48", "Uf48", "Vf48", "Wf48"]
HUR_REL    = [1e-3, 2e-3, 3e-3, 4e-3, 6e-3]
HUR_PARAMS, HUR_EPOCHS = 6000, 30

tgt = np.fromfile(HUR_DIR + HUR_TARGET + ".bin.f32", dtype=np.float32).reshape(HUR_SHAPE)
aux = [np.memmap(HUR_DIR + a + ".bin.f32", dtype=np.float32, mode="r", shape=HUR_SHAPE)
       for a in HUR_AUX]
r = bench_field("Hurricane/CLOUD", tgt, HUR_DIR + HUR_TARGET + ".bin.f32", aux, HUR_SHAPE,
                HUR_REL, HUR_PARAMS, HUR_EPOCHS)
del tgt, aux
torch.cuda.empty_cache() if torch.cuda.is_available() else None

fig, ax = plt.subplots(figsize=(7, 5.5))
plot_panel(ax, r, "Hurricane — CLOUD (+aux)")
fig.suptitle("Hurricane 100×500×500 CLOUD — SZ3+model vs SPERR  (6k params, +aux)", fontsize=12)
plt.tight_layout()
plt.savefig("sperr_cmp_hurricane.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: sperr_cmp_hurricane.png")

## 4. Magnetic Reconnection 512³  —  single field (30k params)

In [ ]:
MAG_FILE  = "/home/sam/Halo_Finder/halo_finder_v1/magnetic_reconnection_512x512x512_float32.raw"
MAG_SHAPE = (512, 512, 512)
MAG_REL   = [1e-3, 3e-3, 5e-3, 7e-3]
MAG_PARAMS, MAG_EPOCHS = 30000, 10

mag = np.fromfile(MAG_FILE, dtype=np.float32).reshape(MAG_SHAPE)
r = bench_field("Magnetic", mag, MAG_FILE, [], MAG_SHAPE, MAG_REL, MAG_PARAMS, MAG_EPOCHS)
del mag
torch.cuda.empty_cache() if torch.cuda.is_available() else None

fig, ax = plt.subplots(figsize=(7, 5.5))
plot_panel(ax, r, "Magnetic Reconnection 512³ — single field")
fig.suptitle("Magnetic Reconnection 512³ — SZ3+model vs SPERR  (30k params)", fontsize=12)
plt.tight_layout()
plt.savefig("sperr_cmp_magnetic.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: sperr_cmp_magnetic.png")

In [ ]:
# ── wpx 256×256×2048 (double) ────────────────────────────────────────────────
WPX_RAW       = "/home/sam/Halo_Finder/halo_finder_v1/wpx-256_256_2048_double.raw"
WPX_SRC_SHAPE = (256, 256, 2048)        # as stored (float64)

# double -> float32 (this notebook trains/compresses in f32); reorient so the
# model's 2-D slices are SQUARE (256×256): move the 2048 axis to depth.
_wpx = np.fromfile(WPX_RAW, dtype=np.float64).reshape(WPX_SRC_SHAPE).astype(np.float32)
_wpx = np.ascontiguousarray(np.transpose(_wpx, (2, 0, 1)))   # -> (2048, 256, 256)
WPX_SHAPE = _wpx.shape

# SPERR reads a file with --ftype 32, so write the reoriented f32 volume for it
WPX_F32 = "/tmp/wpx_2048_256_256_f32.raw"
_wpx.tofile(WPX_F32)

WPX_REL              = [1e-3, 3e-3, 5e-3, 7e-3, 9e-3]
WPX_PARAMS, WPX_EPOCHS = 30000, 10        # depth=2048 → epochs are ~4× Magnetic; lower if slow

r = bench_field("wpx", _wpx, WPX_F32, [], WPX_SHAPE, WPX_REL, WPX_PARAMS, WPX_EPOCHS)
del _wpx
torch.cuda.empty_cache() if torch.cuda.is_available() else None

fig, ax = plt.subplots(figsize=(7, 5.5))
plot_panel(ax, r, "wpx 256×256×2048 (double→f32)")
fig.suptitle("wpx — SZ3+model vs SPERR  (30k params)", fontsize=12)
plt.tight_layout()
plt.savefig("sperr_cmp_wpx.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: sperr_cmp_wpx.png")
